# Skeleton Animatronic: Low-Latency Dialogue Subsystem Demo

This interactive notebook demonstrates the **low-latency context-specific dialogue engine** for the animatronic skeleton.

### Key Capabilities Demonstrated:
1. **Multimodal Sensory Context**: Ingests vision tracking (head angles, subject distance, detected objects) alongside audio transcripts.
2. **Streaming Clause Chunking**: Streams token fragments directly into natural speech boundaries so downstream TTS & mouth sync begin playing on the very first clause without waiting for full completion.
3. **Telemetry & Latency Budget**: Measures **Time-To-First-Token (TTFT)** and **Time-To-First-Chunk (TTFC)** to meet animatronic responsiveness ($<500\text{ ms}$).
4. **Barge-In Interruption Handling**: Instantly cancels in-flight LLM generation and motor queues when the user speaks again.
5. **Phonetic Sanitizer**: Automatically purges stage directions (`*cackles*`, `[whispers]`), asterisks, and emojis that would corrupt TTS pronunciation and lip-sync.

In [ ]:
import asyncio
import os
import sys
import time
from pathlib import Path

# Ensure the skeleton package is on the python path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from skeleton.dialogue.config import DialogueConfig
from skeleton.dialogue.models import DialogueState, SpeechChunk, VisionContext
from skeleton.dialogue.llm_client import LowLatencyLLMClient, MockLLMClient
from skeleton.dialogue.prompt_builder import PromptBuilder
from skeleton.dialogue.sanitizer import TextSanitizer
from skeleton.dialogue.sentence_chunker import SentenceChunker
from skeleton.dialogue.orchestrator import DialogueOrchestrator

print("Skeleton Dialogue Engine successfully imported!")

## 1. Engine Initialization (Mock vs. Live API)

You can run this demo in two modes:
- **Offline Mock Mode (Default)**: Emulates ultra-fast LLM token generation with customizable latency without requiring an API key.
- **Live Cloud Mode**: Uses Groq, Cerebras, or OpenAI-compatible streaming endpoints if `SKELETON_LLM_API_KEY` is set.

In [ ]:
api_key = os.environ.get("SKELETON_LLM_API_KEY")
use_live_api = bool(api_key)

config = DialogueConfig()

if use_live_api:
    print(f"[Mode: Live API] Connecting to {config.base_url} with model {config.model}")
    llm_client = LowLatencyLLMClient(config)
else:
    print("[Mode: Offline Mock] Using MockLLMClient with simulated 35ms TTFT and 15ms inter-token stream.")
    llm_client = MockLLMClient(
        simulated_response="Well, look who decided to wander into my lair. Nice shoes, do they come with a personality?",
        ttft_delay_s=0.035,
        inter_token_delay_s=0.015,
        config=config,
    )

orchestrator = DialogueOrchestrator(config=config, llm_client=llm_client)

## 2. Interactive Multimodal Dialogue & Streaming Telemetry

Let's simulate an interaction where the skeleton sees a visitor holding a coffee cup looking to the right, and the audio receiver transcribes the user's greeting.

In [ ]:
# Define simulated vision tracking state
vision_state = VisionContext(
    subject_detected=True,
    pan_angle_deg=25.0,  # 25 degrees to the right
    tilt_angle_deg=-5.0,
    distance_m=1.8,
    detected_objects=["coffee mug", "headphones"],
    subject_facing_skeleton=False,  # looking away
)

user_speech = "Hey there, can you see what I'm holding?"

print("--- Vision Context Summary ---")
print(vision_state.to_prompt_string())
print("\n--- User Utterance ---")
print(f"User: \"{user_speech}\"")

In [ ]:
async def run_dialogue_stream(user_text, vision):
    print("\n--- Live Chunk Stream & Latency Telemetry ---")
    chunk_count = 0
    first_chunk_ms = None
    
    async for chunk in orchestrator.process_utterance(user_text, vision):
        chunk_count += 1
        if first_chunk_ms is None:
            first_chunk_ms = chunk.elapsed_ms
        
        tag = "[FINAL]" if chunk.is_final else f"[CHUNK {chunk.sequence_index}]"
        print(f"{tag} (+{chunk.elapsed_ms:6.1f}ms | {chunk.word_count:2d} words): \"{chunk.text}\"")
    
    print("--------------------------------------------")
    print(f"Time to First Chunk (TTS ready): {first_chunk_ms:.1f} ms")
    print(f"Total Chunks Emitted: {chunk_count}")

# Execute dialogue in notebook event loop
await run_dialogue_stream(user_speech, vision_state)

## 3. Barge-In Interruption Testbed

When a user interrupts while the skeleton is speaking, the orchestrator cancels the active task and immediately begins processing the new utterance without motor thrashing or double-speaking.

In [ ]:
# Configure a slow streaming mock to simulate the skeleton speaking a longer remark
slow_mock = MockLLMClient(
    simulated_response="I was in the middle of a very witty soliloquy about calcium when you rudely barged in.",
    ttft_delay_s=0.03,
    inter_token_delay_s=0.08,
)
barge_in_orchestrator = DialogueOrchestrator(config=config, llm_client=slow_mock)

print("[1] User speaks: 'Tell me a story...'")
stream = barge_in_orchestrator.process_utterance("Tell me a story")

# Receive the first chunk
first_chunk = await anext(stream)
print(f"Skeleton started speaking: \"{first_chunk.text}\"")

print("\n[2] User abruptly interrupts: 'Stop, never mind!'")
# Interruption triggered by new utterance call
slow_mock.simulated_response = "Fine, I have better things to haunt anyway."
slow_mock.inter_token_delay_s = 0.01

async for chunk in barge_in_orchestrator.process_utterance("Stop, never mind!"):
    print(f"Interrupted Recovery Chunk: \"{chunk.text}\"")

print("\nBarge-In successfully handled with clean cancellation!")

## 4. Phonetic & Lip-Sync Sanitizer Inspection

Verifying that stage directions, asterisks, and emojis are stripped so the physical mouth servo doesn't desynchronize or speak 'asterisk'.

In [ ]:
raw_llm_outputs = [
    "*cackles dryly* You really think you can defeat me? 💀",
    "[whispering softly] Don't look behind you, fleshy.",
    "I have **two hundred and six** bones, and `zero` patience.",
    "Oh please... (clears non-existent throat) you call that dancing? 😂"
]

print("RAW MODEL OUTPUT -> SANITIZED FOR TTS & LIP-SYNC")
print("=" * 60)
for raw in raw_llm_outputs:
    clean = TextSanitizer.sanitize(raw)
    print(f"Raw:   {raw}")
    print(f"Clean: {clean}")
    print("-" * 60)